### 04-function-calling

In [ ]:
from rag_helper import RAGBase, load_faq_data
from minsearch import Index
from openai import OpenAI

documents = load_faq_data()

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)


from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
import os


model_name = "models/gemini-3.1-flash-lite"

client = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


assistant = RAGBase(
    index=index,
    llm_client=client,
    )

In [4]:
assistant.rag('How do I run Docker on Windows?')

"I don't know."

In [5]:
assistant = RAGBase(
    index=index,
    llm_client=client,
    course = 'mlops-zoomcamp'
    )

In [7]:
print(assistant.rag('How do I run Docker on Windows?'))

If you want to install Docker in WSL2 on Windows without Docker Desktop, follow these steps:

1. **Install Docker**
   You can ignore the warnings during installation.
   ```bash
   curl -fsSL https://get.docker.com -o get-docker.sh
   sudo sh get-docker.sh
   ```

2. **Add Your User to the Docker Group**
   ```bash
   sudo usermod -aG docker $USER
   ```

3. **Enable the Docker Service**
   ```bash
   sudo systemctl enable docker.service
   ```

4. **Test the Installation**
   Verify that both Docker and Docker Compose are installed successfully.
   ```bash
   docker --version
   docker compose version
   docker run hello-world
   ```

5. **Ensure Docker Starts Automatically**
   If the service does not start automatically after restarting WSL, update your `.profile` or `.zprofile` file with:
   ```bash
   if grep -q "microsoft" /proc/version > /dev/null 2>&1; then
      if service docker status 2>&1 | grep -q "is not running"; then
         wsl.exe --distribution "${WSL_DISTRO_NAME}"

In [8]:
print(assistant.rag('How do I run ducker on Windows?'))

I don't know.


In [9]:
### Спросим без инструкций 

In [ ]:
client.chat

In [19]:
message_history = [
    #{'role': 'system', 'content': self.instructions},
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]


response = client.chat.completions.create(
    model='models/gemini-3.1-flash-lite',
    messages= message_history,
)

print(response.choices[0].message.content)


To give you the correct answer, I need a little more information, as I am an AI assistant and not a specific course instructor.

**Could you please clarify:**

1.  **What is the name of the course** and what platform is it on (e.g., Coursera, Udemy, a specific university website, or a private workshop)?
2.  **Is it a "Self-Paced" course or a "Cohort-Based" course?**
    *   **Self-paced:** You can almost always join these at any time.
    *   **Cohort-based:** These usually have specific start and end dates. If the course has already started or finished, you might need to wait for the next "session" or "intake" to open.

---

### General steps you can take right now:

*   **Check the Course Page:** Look for a button that says "Enroll," "Register," or "Join Now." If the button is greyed out or says "Join Waitlist," enrollment is likely closed.
*   **Check for "Late Enrollment":** Sometimes, even if a course has started, you can still join during the first week. Look for any notices abou

In [21]:
## Нам нужно рассказать модели о нашей функции поиска. 
## Модель этого не делает см. наш код Python. Он видит только схему, о
# описывающую, что что делает функция и какие аргументы она принимает.

In [22]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

- type- это инструмент функционального типа (API также поддерживает другие типы инструментов)
- name- имя функции. Когда модель вызвать ее, это ссылается на это имя
- description- это важно. Модель читает его, чтобы решить когда вызывать функцию. Хорошее описание помогает модели сделать лучшие решения
- parameters- схема JSON для аргументов функции. Модель генерирует аргументы, соответствующие этой схеме
- additionalProperties: False- сообщает модели, что она может использовать только поля, которые мы перечислили, без дополнительных

### Отправка запроса с помощью инструмента

In [24]:
search_tool = {
    "type": "function",
    "function": {  # <--- Не хватало этого ключа-обертки
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [25]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.
If you look up information, use FAQ search.
""".strip()

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}
]

response = client.chat.completions.create(
    model='models/gemini-3.1-flash-lite',
    messages= chat_messages,
    tools=[search_tool],
)

print(response.choices[0].message.content)

None


In [30]:
message = response.choices[0].message

if message.tool_calls:
    print("Модель запросила вызов инструмента (Tool Call):")
    for tool_call in message.tool_calls:
        print(f"Название функции: {tool_call.function.name}")
        print(f"Аргументы (JSON): {tool_call.function.arguments}")
else:
    print("Модель ответила текстом:")
    print(message.content)

Модель запросила вызов инструмента (Tool Call):
Название функции: search
Аргументы (JSON): {"query":"can I join the course late"}


In [31]:
def search(query):
    # Предполагается, что переменная `index` (ваш minsearch.Index) доступна глобально
    boost = {'question': 3.0, 'section': 0.5}
    
    results = index.search(
        query=query,
        filter_dict={'course': 'llm-zoomcamp'}, # Опциональный фильтр
        boost_dict=boost,
        num_results=3 # Берем топ-3 документа
    )
    return results

In [32]:
import json

# 1. Достаем аргументы вызова функции из ответа API (стандартный путь)
call = response.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)

In [33]:
args

{'query': 'can I join the course late'}

In [34]:
# 2. Вызываем нашу функцию поиска
results = search(**args)
result_json = json.dumps(results, indent=2)

In [36]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [37]:
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\nIf you look up information, use FAQ search."},
 {'role': 'user',
  'content': 'I just discovered the course. Can I still join it?'}]

In [38]:
# 3. ДОБАВЛЯЕМ В ИСТОРИЮ ответ модели (обязательный шаг, иначе API выдаст ошибку)
chat_messages.append(response.choices[0].message)
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\nIf you look up information, use FAQ search."},
 {'role': 'user',
  'content': 'I just discovered the course. Can I still join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='V979Gnsw', function=Function(arguments='{"query":"can I join the course late"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'EjQKMgEMOdbHExqDRjJHQZkOttN/kjT1yk5d6Bq/btVqIZbM0mLVBYgHyTc2vl3JNhCe61x1'}})])]

In [39]:
# 4. ДОБАВЛЯЕМ В ИСТОРИЮ результат выполнения функции (роль 'tool')
chat_messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "name": call.function.name,
    "content": result_json,
})
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\nIf you look up information, use FAQ search."},
 {'role': 'user',
  'content': 'I just discovered the course. Can I still join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='V979Gnsw', function=Function(arguments='{"query":"can I join the course late"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'EjQKMgEMOdbHExqDRjJHQZkOttN/kjT1yk5d6Bq/btVqIZbM0mLVBYgHyTc2vl3JNhCe61x1'}})]),
 {'role': 'tool',
  'tool_call_id': 'V979Gnsw',
  'name': 'search',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you wan

In [40]:
# 5. Делаем финальный запрос к Gemini, чтобы он прочитал result_json и ответил
final_response = client.chat.completions.create(
    model='models/gemini-3.1-flash-lite',
    messages=chat_messages,
    tools=[search_tool],
)

In [41]:
# Выводим финальный ответ
print(final_response.choices[0].message.content)

Yes, you can still join!

You don't need a formal registration to start—you can begin watching the materials and submitting your homework right away.

However, please keep two important things in mind regarding the certificate:
1. **Deadlines:** If you intend to earn a certificate, you must submit your project while submissions are still being accepted.
2. **Cohort/Live Mode:** Certificates are only awarded to those who complete the course with a "live" cohort. This is because you are required to peer-review others' projects, which can only be done while the course is running. The self-paced mode does not qualify for a certificate.


### 05-agentic-loop

На предыдущем уроке мы выполняли вызов функций вручную: отправляли сообщение, получить вызов функции, запустить его, отправить результат, получить ответ. Это работает для одного вызова функции. Но что, если модель захочет сделать множественные поиски? Что делать, если первый поиск не дал ответа?

Нам нужна петля. Агент — это именно то, что нужно — цикл, который постоянно вызывает модель, выполнение инструментов и отправка результатов обратно в модель сделано.

In [43]:
#### A stronger developer prompt

In [ ]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

####  A generic function-call helper

Вместо жесткого кодирования функции поиска давайте создадим помощника, который считывает имя функции из выходных данных модели и вызывает сопоставление Функция Python:

In [44]:
# def make_call(call):
#     args = json.loads(call.arguments)
#     f_name = call.name
#     f = globals()[f_name]
#     result = f(**args)
#     result_json = json.dumps(result, indent=2)
#     return {
#         "type": "function_call_output",
#         'call_id': call.call_id,
#         'output': result_json,
#     }

import json

# Реестр доступных функций (безопасный аналог globals)
AVAILABLE_TOOLS = {
    "search": search 
}

def make_call(call):
    """
    Принимает объект tool_call от модели, выполняет нужную Python-функцию 
    и возвращает результат в строгом формате, который ожидает API.
    """
    # 1. Извлекаем данные по стандарту OpenAI
    args = json.loads(call.function.arguments)
    f_name = call.function.name
    
    # 2. Безопасно достаем функцию из реестра
    if f_name not in AVAILABLE_TOOLS:
        raise ValueError(f"Модель попыталась вызвать неизвестную функцию: {f_name}")
        
    f = AVAILABLE_TOOLS[f_name]
    
    # 3. Выполняем функцию и парсим результат
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    
    # 4. Возвращаем словарь в правильном формате (role: tool)
    return {
        "role": "tool",
        "tool_call_id": call.id,
        "name": f_name,
        "content": result_json,
    }

In [47]:
def agent_loop(question, model='models/gemini-3.1-flash-lite',):
    chat_messages = [
        {'role': 'developer', 'content': developer_prompt},
        {'role': 'user', 'content': question}
    ]
    # Запускаем бесконечный цикл агента
    while True:
        print("🤖 Агент думает...")
        
        # 1. Отправляем текущую историю сообщений в модель
        response = client.chat.completions.create(
            model='models/gemini-3.1-flash-lite',
            messages=chat_messages,
            tools=[search_tool],
        )

        # 2. Извлекаем ответ модели
        message = response.choices[0].message

        # 3. Обязательно добавляем ответ модели в историю диалога
        chat_messages.append(message)

        # 4. Проверяем, есть ли запросы на вызов функций
        if message.tool_calls:
            # Модель решила, что ей нужны данные. Обрабатываем каждый вызов
            for tool_call in message.tool_calls:
                print(f"🛠️  Выполняю функцию: {tool_call.function.name}")
                print(f"   Аргументы: {tool_call.function.arguments}")
                
                # Используем ваш хелпер make_call для выполнения Python-кода
                tool_result = make_call(tool_call)
                
                # Добавляем результат обратно в историю
                chat_messages.append(tool_result)
                
            # После обработки всех функций цикл while начнется заново.
            # Модель получит результаты и решит, нужен ли еще один поиск, 
            # или можно уже отвечать пользователю.
                
        else:
            # 5. Если tool_calls пустой, значит модель сгенерировала финальный текст
            print("\n✅ Финальный ответ модели:")
            print(message.content)
            
            # Прерываем цикл, работа агента завершена
            break

In [48]:
agent_loop('How do I run ducker on windows?')

🤖 Агент думает...
🛠️  Выполняю функцию: search
   Аргументы: {"query":"run docker on windows"}
🤖 Агент думает...

✅ Финальный ответ модели:
To run Docker on Windows, the recommended approach is to install **Docker Desktop**. Here are the general steps to get started:

1.  **Download and Install:** Visit the official [Docker website](https://www.docker.com/products/docker-desktop/) and download Docker Desktop for Windows.
2.  **Enable WSL 2:** Docker Desktop for Windows works best with the Windows Subsystem for Linux (WSL 2) backend. During installation, ensure the option to use WSL 2 is checked. You may need to enable the "Windows Subsystem for Linux" feature in your Windows settings if it isn't already enabled.
3.  **Launch Docker:** Once installed, launch Docker Desktop. It will run as a background service.
4.  **Verify:** Open your terminal (PowerShell, Command Prompt, or your preferred terminal) and run the following command to verify it is working correctly:
    ```bash
    docker

In [49]:
agent_loop('I just discovered the course. Can I still join it?')

🤖 Агент думает...
🛠️  Выполняю функцию: search
   Аргументы: {"query":"can I still join the course"}
🤖 Агент думает...

✅ Финальный ответ модели:
Yes, you can still join!

However, please keep in mind the following regarding the certificate:

*   **Submissions:** If you intend to earn a certificate, you must submit your final project while the course is still accepting submissions.
*   **Live Cohort:** Certificates are only awarded to students who complete the course with a "live" cohort. This is because the certification process requires peer-reviewing other students' capstone projects, which can only be done while the course is actively running.
*   **Homework:** Homework is not mandatory for receiving a certificate (it is recommended for practice and for your rank on the leaderboard), but passing the capstone project is required.


In [50]:
agent_loop('Is Gemini better than ChatGPT?')

🤖 Агент думает...

✅ Финальный ответ модели:
Whether Gemini is "better" than ChatGPT depends entirely on what you are trying to accomplish. Both are powerful Large Language Models (LLMs), but they have different strengths, ecosystems, and architectural focuses.

Here is a breakdown to help you decide which one might be better for your specific needs:

### 1. Ecosystem Integration
*   **Gemini:** If you live in the Google ecosystem, Gemini is often superior. It integrates directly with Google Workspace (Docs, Gmail, Drive, Maps, YouTube). It can summarize your emails, pull information from your documents, or find locations in Maps.
*   **ChatGPT:** If you use OpenAI’s API for development, or if you prefer a standalone platform that has built a massive library of user-created "GPTs" (customized versions of ChatGPT for specific tasks), ChatGPT offers a more mature plugin and customization ecosystem.

### 2. Context Window (The Amount of Information it can process)
*   **Gemini:** Gemini (

In [51]:
agent_loop('Who won World Cup in 1950?')

🤖 Агент думает...
🛠️  Выполняю функцию: search
   Аргументы: {"query":"Who won the 1950 World Cup?"}
🤖 Агент думает...

✅ Финальный ответ модели:
Uruguay won the 1950 FIFA World Cup, defeating Brazil in the final match.
